# INF201 – Deliverable 2: Campus Coffee Shop Sales Pipeline

**Group members:** Mathias Gundersen, Jakub Mermer  
**GitHub repository:** https://github.com/saltonese/Del_2-G6_5.git

## 1. Introduction and approach

This notebook implements an ETL pipeline for the cash-register logs of the Campus Coffee chain:

1. **Extract** – recursively find all `.log` files, read them with the correct encoding (UTF-8 or ISO-8859-1) and pick out the valid sales lines with a regular expression.
2. **Transform** – build a DataFrame, remove price glitches with NumPy, and correct NOK prices that were registered on the EUR scale (× 10).
3. **Load / report** – hourly revenue per item, a status pivot table and refund ratios, written to `output/`.

Tasks 1–4 are small, individually tested building blocks. Task 5 connects them together into one .

### Interpretation choices

The assignment leaves a few things open. These are the choices made here:


### How to run
1. `python generate_data_for_d2.py` (creates `data/`; `BASE_DIR` was changed to `Path("Data")` and `random.seed(42)` added so all group members get identical data).
2. Run this notebook from the same folder. Output files are written to `output/`.

In [1]:
import json
import re
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print(Path.cwd())
DATA_DIR = Path.cwd().parent / "Data"
OUTPUT_DIR = Path("output")

EPSILON = 1e-6          # keeps the refund ratio finite when sales_count == 0
NOK_CORRECTION = 10.0   # NOK prices were registered on the EUR scale
OUTLIER_N_STD = 3.0     # |x - mean| > 3 * std  ->  hardware glitch

print(f"pandas {pd.__version__} | numpy {np.__version__}")
assert DATA_DIR.exists(), "Run `python generate_data_for_d2.py` first to create the Data/ folder."

c:\Users\mathi\StudieNMBU\PROG\Del_2-G6_5\Notebooks
pandas 3.1.0 | numpy 2.5.4
